<a href="https://colab.research.google.com/github/Thomas5040/AI-Powered-Personalized-Food-Recommendation-Analytics-System/blob/main/02_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd

file_path = "Food_Recommendation_100_Indian_Foods.xlsx"

df = pd.read_excel(file_path)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset Shape: (100, 16)

Columns:
['Food_ID', 'Food_Name', 'Category', 'Serving_Size', 'Calories', 'Protein_g', 'Carbohydrates_g', 'Sugar_g', 'Total_Fat_g', 'Saturated_Fat_g', 'Fiber_g', 'Sodium_mg', 'Dietary_Type', 'Protein_Level', 'Health_Profile', 'Ingredients']

First 5 rows:


,Food_ID,Food_Name,Category,Serving_Size,Calories,Protein_g,Carbohydrates_g,Sugar_g,Total_Fat_g,Saturated_Fat_g,Fiber_g,Sodium_mg,Dietary_Type,Protein_Level,Health_Profile,Ingredients
0,V001,Idli with Sambar,Breakfast,3 idli + 150 g sambar,360,12,62,6,6,1.2,8,650,Vegetarian,High,Balanced,Vegetarian ingredients; exact preparation may ...
1,V002,Masala Dosa,Breakfast,1 large dosa,420,9,58,5,17,3.5,4,720,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...
2,V003,Plain Dosa with Sambar,Breakfast,2 dosa + 150 g sambar,390,11,64,5,8,1.5,6,680,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...
3,V004,Vegetable Upma,Breakfast,250 g,280,7,45,4,9,1.5,6,520,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...
4,V005,Vegetable Poha,Breakfast,250 g,300,7,48,4,8,1.2,5,480,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...


In [9]:
print("Vegetarian:", (df["Dietary_Type"] == "Vegetarian").sum())
print("Non-Vegetarian:", (df["Dietary_Type"] == "Non-Vegetarian").sum())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Vegetarian: 50
Non-Vegetarian: 50

Missing Values:
Food_ID            0
Food_Name          0
Category           0
Serving_Size       0
Calories           0
Protein_g          0
Carbohydrates_g    0
Sugar_g            0
Total_Fat_g        0
Saturated_Fat_g    0
Fiber_g            0
Sodium_mg          0
Dietary_Type       0
Protein_Level      0
Health_Profile     0
Ingredients        0
dtype: int64

Duplicate Rows: 0


In [10]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [11]:
file_path = "Food_Recommendation_100_Indian_Foods.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset Shape: (100, 16)


,Food_ID,Food_Name,Category,Serving_Size,Calories,Protein_g,Carbohydrates_g,Sugar_g,Total_Fat_g,Saturated_Fat_g,Fiber_g,Sodium_mg,Dietary_Type,Protein_Level,Health_Profile,Ingredients
0,V001,Idli with Sambar,Breakfast,3 idli + 150 g sambar,360,12,62,6,6,1.2,8,650,Vegetarian,High,Balanced,Vegetarian ingredients; exact preparation may ...
1,V002,Masala Dosa,Breakfast,1 large dosa,420,9,58,5,17,3.5,4,720,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...
2,V003,Plain Dosa with Sambar,Breakfast,2 dosa + 150 g sambar,390,11,64,5,8,1.5,6,680,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...
3,V004,Vegetable Upma,Breakfast,250 g,280,7,45,4,9,1.5,6,520,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...
4,V005,Vegetable Poha,Breakfast,250 g,300,7,48,4,8,1.2,5,480,Vegetarian,Medium,Balanced,Vegetarian ingredients; exact preparation may ...


In [12]:
print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nDietary Type:")
print(df["Dietary_Type"].value_counts())


Column Names:
['Food_ID', 'Food_Name', 'Category', 'Serving_Size', 'Calories', 'Protein_g', 'Carbohydrates_g', 'Sugar_g', 'Total_Fat_g', 'Saturated_Fat_g', 'Fiber_g', 'Sodium_mg', 'Dietary_Type', 'Protein_Level', 'Health_Profile', 'Ingredients']

Missing Values:
Food_ID            0
Food_Name          0
Category           0
Serving_Size       0
Calories           0
Protein_g          0
Carbohydrates_g    0
Sugar_g            0
Total_Fat_g        0
Saturated_Fat_g    0
Fiber_g            0
Sodium_mg          0
Dietary_Type       0
Protein_Level      0
Health_Profile     0
Ingredients        0
dtype: int64

Duplicate Rows: 0

Dietary Type:
Dietary_Type
Vegetarian        50
Non-Vegetarian    50
Name: count, dtype: int64


In [13]:
df = df.drop_duplicates().reset_index(drop=True)

# Clean text columns
text_columns = [
    "Food_ID",
    "Food_Name",
    "Category",
    "Serving_Size",
    "Dietary_Type",
    "Protein_Level",
    "Health_Profile",
    "Ingredients"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# Clean dietary type
df["Dietary_Type"] = df["Dietary_Type"].str.title()

# Clean category
df["Category"] = df["Category"].str.title()

In [14]:
nutrition_features = [
    "Calories",
    "Protein_g",
    "Carbohydrates_g",
    "Sugar_g",
    "Total_Fat_g",
    "Saturated_Fat_g",
    "Fiber_g",
    "Sodium_mg"
]

# Convert nutrition columns to numeric
for col in nutrition_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows with missing nutrition values
df = df.dropna(subset=nutrition_features).reset_index(drop=True)

print("\nFinal Dataset Shape:", df.shape)



Final Dataset Shape: (100, 16)


In [15]:
print("\nNegative Values:")

negative_values = (df[nutrition_features] < 0).sum()

display(negative_values)

print("\nSaturated Fat Greater Than Total Fat:")

invalid_fat = df[df["Saturated_Fat_g"] > df["Total_Fat_g"]]

print("Invalid Rows:", len(invalid_fat))


Negative Values:


,0
Calories,0
Protein_g,0
Carbohydrates_g,0
Sugar_g,0
Total_Fat_g,0
Saturated_Fat_g,0
Fiber_g,0
Sodium_mg,0



Saturated Fat Greater Than Total Fat:
Invalid Rows: 0


In [16]:
X = df[nutrition_features]

print("\nFeatures used for K-Means:")
print(nutrition_features)



Features used for K-Means:
['Calories', 'Protein_g', 'Carbohydrates_g', 'Sugar_g', 'Total_Fat_g', 'Saturated_Fat_g', 'Fiber_g', 'Sodium_mg']


In [17]:
standard_scaler = StandardScaler()

X_scaled = standard_scaler.fit_transform(X)

print("\nStandardScaler completed.")

print("Scaled data shape:", X_scaled.shape)


StandardScaler completed.
Scaled data shape: (100, 8)


In [18]:
print("\nTesting different K values...")

silhouette_scores = {}

for k in range(2, 8):

    temp_model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    temp_labels = temp_model.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, temp_labels)

    silhouette_scores[k] = score

    print(f"K = {k} | Silhouette Score = {score:.4f}")


Testing different K values...
K = 2 | Silhouette Score = 0.2763
K = 3 | Silhouette Score = 0.3112
K = 4 | Silhouette Score = 0.2806
K = 5 | Silhouette Score = 0.2370
K = 6 | Silhouette Score = 0.2972
K = 7 | Silhouette Score = 0.2413


In [19]:
best_k = max(
    silhouette_scores,
    key=silhouette_scores.get
)

print("\nBest K according to Silhouette Score:", best_k)

k = 4


Best K according to Silhouette Score: 3


In [20]:
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df["Food_Cluster"] = kmeans.fit_predict(X_scaled)

print("\nK-Means training completed.")

print("\nNumber of foods in each cluster:")
print(df["Food_Cluster"].value_counts().sort_index())


K-Means training completed.

Number of foods in each cluster:
Food_Cluster
0    28
1    42
2    17
3    13
Name: count, dtype: int64


In [21]:
cluster_summary = df.groupby("Food_Cluster")[nutrition_features].mean()

print("\nCluster Nutrition Summary:")
display(cluster_summary.round(2))


Cluster Nutrition Summary:


,Calories,Protein_g,Carbohydrates_g,Sugar_g,Total_Fat_g,Saturated_Fat_g,Fiber_g,Sodium_mg
Food_Cluster,,,,,,,,
0,497.14,27.36,60.21,4.93,16.82,4.21,4.46,865.00
1,357.62,13.48,48.76,5.93,11.50,2.10,7.88,533.10
2,306.12,34.06,8.59,2.24,14.76,3.61,2.06,472.06
3,510.77,31.08,34.77,4.54,27.31,9.08,3.69,884.62


In [22]:
cluster_profiles = {}

for cluster in cluster_summary.index:

    row = cluster_summary.loc[cluster]

    if (
        row["Protein_g"] >= cluster_summary["Protein_g"].median()
        and row["Fiber_g"] >= cluster_summary["Fiber_g"].median()
    ):
        profile = "Protein & Fiber-Rich Foods"

    elif row["Sugar_g"] >= cluster_summary["Sugar_g"].median():
        profile = "Higher-Sugar Foods"

    elif row["Sodium_mg"] >= cluster_summary["Sodium_mg"].median():
        profile = "Higher-Sodium Foods"

    elif row["Total_Fat_g"] >= cluster_summary["Total_Fat_g"].median():
        profile = "Higher-Fat Foods"

    else:
        profile = "Balanced Foods"

    cluster_profiles[cluster] = profile

In [23]:
df["Food_Profile"] = df["Food_Cluster"].map(cluster_profiles)

print("\nCluster Profiles:")

for cluster, profile in cluster_profiles.items():
    print(f"Cluster {cluster}: {profile}")


Cluster Profiles:
Cluster 0: Higher-Sugar Foods
Cluster 1: Higher-Sugar Foods
Cluster 2: Balanced Foods
Cluster 3: Higher-Sodium Foods


In [24]:
print("\nSample Foods from Each Cluster:")

for cluster in sorted(df["Food_Cluster"].unique()):

    print("\n--------------------------------")
    print(f"Cluster {cluster}")
    print(f"Profile: {cluster_profiles[cluster]}")
    print("--------------------------------")

    sample = df[df["Food_Cluster"] == cluster][
        ["Food_Name", "Dietary_Type", "Protein_g", "Fiber_g",
         "Sugar_g", "Sodium_mg", "Food_Profile"]
    ].head(5)

    display(sample)


Sample Foods from Each Cluster:

--------------------------------
Cluster 0
Profile: Higher-Sugar Foods
--------------------------------


,Food_Name,Dietary_Type,Protein_g,Fiber_g,Sugar_g,Sodium_mg,Food_Profile
1,Masala Dosa,Vegetarian,9,4,5,720,Higher-Sugar Foods
20,Mushroom Biryani,Vegetarian,13,7,5,740,Higher-Sugar Foods
24,Vegetable Fried Rice,Vegetarian,10,5,5,980,Higher-Sugar Foods
46,Paneer Sandwich,Vegetarian,24,6,6,720,Higher-Sugar Foods
50,Chicken Biryani,Non-Vegetarian,31,3,5,980,Higher-Sugar Foods



--------------------------------
Cluster 1
Profile: Higher-Sugar Foods
--------------------------------


,Food_Name,Dietary_Type,Protein_g,Fiber_g,Sugar_g,Sodium_mg,Food_Profile
0,Idli with Sambar,Vegetarian,12,8,6,650,Higher-Sugar Foods
2,Plain Dosa with Sambar,Vegetarian,11,6,5,680,Higher-Sugar Foods
3,Vegetable Upma,Vegetarian,7,6,4,520,Higher-Sugar Foods
4,Vegetable Poha,Vegetarian,7,5,4,480,Higher-Sugar Foods
5,Ven Pongal,Vegetarian,10,4,3,620,Higher-Sugar Foods



--------------------------------
Cluster 2
Profile: Balanced Foods
--------------------------------


,Food_Name,Dietary_Type,Protein_g,Fiber_g,Sugar_g,Sodium_mg,Food_Profile
43,Mixed Nuts,Vegetarian,10,4,2,5,Balanced Foods
63,Chicken Tikka,Non-Vegetarian,43,1,2,680,Balanced Foods
64,Tandoori Chicken,Non-Vegetarian,55,1,2,720,Balanced Foods
65,Grilled Chicken Breast,Non-Vegetarian,62,0,0,180,Balanced Foods
73,Egg Omelette,Non-Vegetarian,21,0,1,240,Balanced Foods



--------------------------------
Cluster 3
Profile: Higher-Sodium Foods
--------------------------------


,Food_Name,Dietary_Type,Protein_g,Fiber_g,Sugar_g,Sodium_mg,Food_Profile
9,Paneer Bhurji,Vegetarian,23,3,5,620,Higher-Sodium Foods
10,Paneer Tikka,Vegetarian,28,3,5,580,Higher-Sodium Foods
11,Palak Paneer,Vegetarian,20,5,5,650,Higher-Sodium Foods
12,Paneer Curry with Roti,Vegetarian,24,9,7,720,Higher-Sodium Foods
51,Mutton Biryani,Non-Vegetarian,28,3,4,1050,Higher-Sodium Foods


In [25]:
recommendation_features = [
    "Protein_g",
    "Fiber_g",
    "Sugar_g",
    "Saturated_Fat_g",
    "Sodium_mg"
]

recommendation_scaler = MinMaxScaler()

recommendation_scaled = recommendation_scaler.fit_transform(
    df[recommendation_features]
)

recommendation_df = pd.DataFrame(
    recommendation_scaled,
    columns=[
        "Protein_Score",
        "Fiber_Score",
        "Sugar_Score",
        "Saturated_Fat_Score",
        "Sodium_Score"
    ],
    index=df.index
)

df = pd.concat(
    [df, recommendation_df],
    axis=1
)


In [26]:
df["Low_Sugar_Score"] = 1 - df["Sugar_Score"]

df["Low_Saturated_Fat_Score"] = (
    1 - df["Saturated_Fat_Score"]
)

df["Low_Sodium_Score"] = (
    1 - df["Sodium_Score"]
)


In [27]:
df["Recommendation_Score"] = (

    0.30 * df["Protein_Score"]

    + 0.25 * df["Fiber_Score"]

    + 0.20 * df["Low_Sugar_Score"]

    + 0.15 * df["Low_Saturated_Fat_Score"]

    + 0.10 * df["Low_Sodium_Score"]

) * 100


In [28]:
def recommendation_category(score):

    if score >= 75:
        return "Highly Recommended"

    elif score >= 50:
        return "Recommended"

    else:
        return "Less Recommended"


df["Recommendation_Level"] = (
    df["Recommendation_Score"]
    .apply(recommendation_category)
)


In [29]:
def get_recommendations(
    data,
    high_protein=False,
    high_fiber=False,
    low_sugar=False,
    low_sodium=False,
    low_saturated_fat=False,
    dietary_type="All",
    category="All",
    top_n=10
):

    result = data.copy()

    # Dietary filter
    if dietary_type != "All":
        result = result[
            result["Dietary_Type"] == dietary_type
        ]

    # Category filter
    if category != "All":
        result = result[
            result["Category"] == category
        ]

    score_columns = []

    if high_protein:
        score_columns.append("Protein_Score")

    if high_fiber:
        score_columns.append("Fiber_Score")

    if low_sugar:
        score_columns.append("Low_Sugar_Score")

    if low_sodium:
        score_columns.append("Low_Sodium_Score")

    if low_saturated_fat:
        score_columns.append(
            "Low_Saturated_Fat_Score"
        )

    # If no preference selected
    if len(score_columns) == 0:

        result["Personalized_Score"] = (
            result["Recommendation_Score"]
        )

    else:

        result["Personalized_Score"] = (
            result[score_columns]
            .mean(axis=1)
            * 100
        )

    result = result.sort_values(
        "Personalized_Score",
        ascending=False
    )

    return result.head(top_n)


In [30]:
def explain_food(row):

    explanation = []

    if row["Protein_g"] >= df["Protein_g"].median():
        explanation.append("good protein")

    if row["Fiber_g"] >= df["Fiber_g"].median():
        explanation.append("good fiber")

    if row["Sugar_g"] <= df["Sugar_g"].median():
        explanation.append("lower sugar")

    if row["Saturated_Fat_g"] <= df["Saturated_Fat_g"].median():
        explanation.append("lower saturated fat")

    if row["Sodium_mg"] <= df["Sodium_mg"].median():
        explanation.append("lower sodium")

    if len(explanation) == 0:
        return "Balanced nutritional profile."

    return "This food is recommended because it has " + \
           ", ".join(explanation) + "."


df["Recommendation_Explanation"] = df.apply(
    explain_food,
    axis=1
)


In [31]:
print("\nTop High-Protein Recommendations:")

test_recommendations = get_recommendations(
    df,
    high_protein=True,
    top_n=10
)

display(
    test_recommendations[
        [
            "Food_Name",
            "Dietary_Type",
            "Category",
            "Protein_g",
            "Fiber_g",
            "Sugar_g",
            "Recommendation_Score",
            "Food_Profile"
        ]
    ].round(2)
)


Top High-Protein Recommendations:


,Food_Name,Dietary_Type,Category,Protein_g,Fiber_g,Sugar_g,Recommendation_Score,Food_Profile
65,Grilled Chicken Breast,Non-Vegetarian,High Protein Meal,62,0,0,71.46,Balanced Foods
64,Tandoori Chicken,Non-Vegetarian,High Protein Meal,55,1,2,58.96,Balanced Foods
88,Grilled Chicken Salad,Non-Vegetarian,High Protein Meal,46,8,5,65.64,Balanced Foods
63,Chicken Tikka,Non-Vegetarian,High Protein Meal,43,1,2,54.18,Balanced Foods
98,Chicken Quinoa Bowl,Non-Vegetarian,High Protein Meal,43,7,5,62.55,Higher-Sugar Foods
66,Chicken Seekh Kebab,Non-Vegetarian,High Protein Meal,42,1,2,48.23,Higher-Sodium Foods
78,Grilled Fish,Non-Vegetarian,High Protein Meal,42,0,1,57.95,Balanced Foods
87,Chicken Salad,Non-Vegetarian,High Protein Meal,42,7,5,62.72,Balanced Foods
84,Chicken Kebab,Non-Vegetarian,High Protein Snack,39,1,2,50.54,Balanced Foods
97,Chicken Caesar Salad,Non-Vegetarian,High Protein Meal,38,5,4,49.83,Higher-Sodium Foods


In [32]:
print("\nTop Vegetarian Recommendations:")

veg_recommendations = get_recommendations(
    df,
    dietary_type="Vegetarian",
    high_protein=True,
    low_sugar=True,
    top_n=10
)

display(
    veg_recommendations[
        [
            "Food_Name",
            "Category",
            "Protein_g",
            "Sugar_g",
            "Fiber_g",
            "Personalized_Score",
            "Food_Profile"
        ]
    ].round(2)
)


Top Vegetarian Recommendations:


,Food_Name,Category,Protein_g,Sugar_g,Fiber_g,Personalized_Score,Food_Profile
10,Paneer Tikka,High Protein Snack,28,5,3,56.29,Higher-Sodium Foods
16,Dal Rice,Rice Meal,18,3,11,53.07,Higher-Sugar Foods
9,Paneer Bhurji,High Protein Meal,23,5,3,51.90,Higher-Sodium Foods
7,Adai with Avocado Chutney,Breakfast,16,3,9,51.32,Higher-Sugar Foods
27,Whole Wheat Roti with Dal,Main Meal,19,4,12,51.17,Higher-Sugar Foods
38,Moong Dal Chilla,High Protein Snack,18,4,8,50.29,Higher-Sugar Foods
41,Tofu Salad,High Protein Snack,21,5,7,50.15,Higher-Sugar Foods
46,Paneer Sandwich,High Protein Snack,24,6,6,50.00,Higher-Sugar Foods
11,Palak Paneer,Main Meal,20,5,5,49.27,Higher-Sodium Foods
26,Dal Tadka with Roti,Main Meal,20,5,12,49.27,Higher-Sugar Foods


In [33]:
print("\nTop Non-Vegetarian Recommendations:")

nonveg_recommendations = get_recommendations(
    df,
    dietary_type="Non-Vegetarian",
    high_protein=True,
    low_sugar=True,
    top_n=10
)

display(
    nonveg_recommendations[
        [
            "Food_Name",
            "Category",
            "Protein_g",
            "Sugar_g",
            "Fiber_g",
            "Personalized_Score",
            "Food_Profile"
        ]
    ].round(2)
)


Top Non-Vegetarian Recommendations:


,Food_Name,Category,Protein_g,Sugar_g,Fiber_g,Personalized_Score,Food_Profile
65,Grilled Chicken Breast,High Protein Meal,62,0,0,100.00,Balanced Foods
64,Tandoori Chicken,High Protein Meal,55,2,1,88.30,Balanced Foods
78,Grilled Fish,High Protein Meal,42,1,0,79.68,Balanced Foods
63,Chicken Tikka,High Protein Meal,43,2,1,77.78,Balanced Foods
66,Chicken Seekh Kebab,High Protein Meal,42,2,1,76.90,Higher-Sodium Foods
84,Chicken Kebab,High Protein Snack,39,2,1,74.27,Balanced Foods
79,Fish Fry,Main Meal,34,1,1,72.66,Balanced Foods
88,Grilled Chicken Salad,High Protein Meal,46,5,8,72.08,Balanced Foods
67,Chicken 65,Snack,36,2,1,71.64,Higher-Sodium Foods
68,Chicken Lollipop,Snack,34,2,1,69.88,Higher-Sodium Foods


In [34]:
joblib.dump(
    kmeans,
    "food_kmeans_model.pkl"
)

joblib.dump(
    standard_scaler,
    "food_scaler.pkl"
)

joblib.dump(
    recommendation_scaler,
    "recommendation_scaler.pkl"
)

print("\nML models and scalers saved successfully!")



ML models and scalers saved successfully!


In [35]:
output_file = "Final_Food_Recommendation_Dataset.xlsx"

df.to_excel(
    output_file,
    index=False
)

print("\nFinal dataset saved as:")
print(output_file)



Final dataset saved as:
Final_Food_Recommendation_Dataset.xlsx


In [36]:
print("\n==========================================")
print("PROJECT TRAINING COMPLETED")
print("==========================================")

print("Total Foods:", len(df))

print(
    "Vegetarian:",
    (df["Dietary_Type"] == "Vegetarian").sum()
)

print(
    "Non-Vegetarian:",
    (df["Dietary_Type"] == "Non-Vegetarian").sum()
)

print("Number of Clusters:", k)

print("\nCluster Profiles:")

for cluster, profile in cluster_profiles.items():
    print(
        f"Cluster {cluster}: {profile}"
    )

print("\nFiles Created:")

print("1. food_kmeans_model.pkl")
print("2. food_scaler.pkl")
print("3. recommendation_scaler.pkl")
print("4. Final_Food_Recommendation_Dataset.xlsx")

print("\nNext Step:")
print("Download these files and update the Streamlit application.")


PROJECT TRAINING COMPLETED
Total Foods: 100
Vegetarian: 50
Non-Vegetarian: 50
Number of Clusters: 4

Cluster Profiles:
Cluster 0: Higher-Sugar Foods
Cluster 1: Higher-Sugar Foods
Cluster 2: Balanced Foods
Cluster 3: Higher-Sodium Foods

Files Created:
1. food_kmeans_model.pkl
2. food_scaler.pkl
3. recommendation_scaler.pkl
4. Final_Food_Recommendation_Dataset.xlsx

Next Step:
Download these files and update the Streamlit application.
